In [1]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib
import warnings

warnings.filterwarnings("ignore")

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
  open_df = df3["Open"]
  volume_df = df3["volume"]

except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)
  open_df = df3.xs("Open", level = 0, axis = 1)
  volume_df = df3.xs("Volume", level = 0, axis = 1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")
open_df = open_df.dropna(axis=1, how = "all")
volume_df = volume_df.dropna(axis =1 , how = "all")

valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns).intersection(open_df.columns).intersection(volume_df.columns)
)
close_df, high_df, low_df, open_df, volume_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
    open_df[valid_cols],
    volume_df[valid_cols]
)




[**************        29%                       ]  146 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[****************      33%                       ]  165 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************59%***                    ]  296 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['HONA', 'FDXF', 'VYLR']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


### Donchian Channels strategy

In [5]:
macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}
obv_dict = {}
obv_ma_dict = {}
bb_upper_dict = {}
bb_middle_dict = {}
bb_lower_dict = {}
dc_upper_dict = {}
dc_middle_dict = {}
dc_lower_dict = {}

donchian_window = 20

for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)
    high_series = high_df[ticker]
    low_series = low_df[ticker]
    volume_values = volume_df[ticker].values

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    obv = talib.OBV(close_df[ticker].to_numpy(dtype = np.float64), volume_df[ticker].to_numpy(dtype = np.float64))
    oma = pd.Series(obv, index=close_df.index).rolling(20).mean()

    upper, middle, lower = talib.BBANDS(
        prices,
        timeperiod=20,
        nbdevup=2,
        nbdevdn=2,
        matype=0
    )

    donchian_upper = high_series.rolling(window=donchian_window).max().shift(1)
    donchian_lower = low_series.rolling(window=donchian_window).min().shift(1)
    donchian_middle = (donchian_upper + donchian_lower) / 2


    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi
    obv_dict[ticker] = obv
    obv_ma_dict[ticker] = oma
    bb_upper_dict[ticker] = upper
    bb_middle_dict[ticker] =  middle
    bb_lower_dict[ticker] = lower
    dc_upper_dict[ticker] = donchian_upper
    dc_lower_dict[ticker] = donchian_lower
    dc_middle_dict[ticker] = donchian_middle

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )
obv_df = pd.DataFrame(obv_dict, index = close_df.index)
oma_df = pd.DataFrame(obv_ma_dict, index = close_df.index)
upper_df = pd.DataFrame(bb_upper_dict, index = close_df.index)
middle_df = pd.DataFrame(bb_middle_dict, index=close_df.index)
lower_df = pd.DataFrame(bb_lower_dict, close_df.index)
donchian_upper_df = pd.DataFrame(dc_upper_dict, index=close_df.index)
donchian_middle_df = pd.DataFrame(dc_middle_dict, index=close_df.index)
donchian_lower_df = pd.DataFrame(dc_lower_dict, index=close_df.index)


entries = (close_df >= donchian_middle_df).shift(1).fillna(False).astype(bool)
exits = (close_df < donchian_middle_df).shift(1).fillna(False).astype(bool)


backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)


print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               219404.659477
Total Return [%]                           119.404659
Benchmark Return [%]                       524.559288
Max Gross Exposure [%]                          100.0
Total Fees Paid                          73512.831186
Max Drawdown [%]                            24.229762
Max Drawdown Duration               592 days 00:00:00
Total Trades                                    71218
Total Closed Trades                             70926
Total Open Trades                                 292
Open Trade PnL                           14012.578866
Win Rate [%]                                39.679384
Best Trade [%]                              361.20906
Worst Trade [%]                            -43.763158
Avg Winning Trade [%]       

### Paired with volatility based measure

In [6]:
entries = ((close_df >= donchian_middle_df) & (close_df >= middle_df)).shift(1).fillna(False).astype(bool)
exits = ((close_df < donchian_middle_df) | (close_df < middle_df)).shift(1).fillna(False).astype(bool)


backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)


print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               205993.681564
Total Return [%]                           105.993682
Benchmark Return [%]                       524.559288
Max Gross Exposure [%]                          100.0
Total Fees Paid                          82095.814128
Max Drawdown [%]                            22.117804
Max Drawdown Duration               578 days 00:00:00
Total Trades                                    75738
Total Closed Trades                             75464
Total Open Trades                                 274
Open Trade PnL                           10818.301372
Win Rate [%]                                39.816336
Best Trade [%]                              361.20906
Worst Trade [%]                            -43.763158
Avg Winning Trade [%]       

### Mixture of multiple strategies along with donchian channels

In [7]:
entries = ((close_df >= middle_df) & (macd_df >= signal_df) & (rsi_df >= 30) & (obv_df >= oma_df ) & (close_df >= donchian_middle_df)).shift(1).fillna(False).astype(bool)
exits = ((close_df < middle_df) | (macd_df < signal_df) | (rsi_df > 70) | (obv_df < oma_df) | (close_df < donchian_middle_df)).shift(1).fillna(False).astype(bool)



backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                            2015-05-05 00:00:00
End                              2026-05-04 00:00:00
Period                            2766 days 00:00:00
Start Value                                 100000.0
End Value                              157088.287815
Total Return [%]                           57.088288
Benchmark Return [%]                      524.559288
Max Gross Exposure [%]                     99.999988
Total Fees Paid                         93308.948103
Max Drawdown [%]                           23.885389
Max Drawdown Duration              520 days 00:00:00
Total Trades                                   68444
Total Closed Trades                            68272
Total Open Trades                                172
Open Trade PnL                           7784.343058
Win Rate [%]                               42.390731
Best Trade [%]                            177.512292
Worst Trade [%]                           -31.391328
Avg Winning Trade [%]                       3.

### MACD & RSI confirmation with Donchian Channels


In [8]:
entries = ((macd_df >= signal_df) & (rsi_df >= 30) & (close_df >= donchian_middle_df)).shift(1).fillna(False).astype(bool)
exits = ((macd_df < signal_df) | (rsi_df > 70) | (close_df < donchian_middle_df)).shift(1).fillna(False).astype(bool)



backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               195651.859297
Total Return [%]                            95.651859
Benchmark Return [%]                       524.559288
Max Gross Exposure [%]                          100.0
Total Fees Paid                          80955.617557
Max Drawdown [%]                            25.157992
Max Drawdown Duration               570 days 00:00:00
Total Trades                                    59657
Total Closed Trades                             59444
Total Open Trades                                 213
Open Trade PnL                            9487.390846
Win Rate [%]                                41.151336
Best Trade [%]                             209.965469
Worst Trade [%]                            -43.763158
Avg Winning Trade [%]       